# Pose predictors

## File Handling
To run predictions a `RobotEnvironment` object and a `HeadsetData` object is needed, those can be loaded from folders or created.

### Creation of RobotEnvironment and HeadsetData
Those 2 datatypes can be created from an GatheredRobotData object and a .vrs file respectively.

In [ ]:
%load_ext autoreload
%autoreload 2
print(__debug__)
from headset_data import *
from robot_environment import *

In [ ]:
robot_data_folder_location = "/home/wmarx/AR-Headset-Localization-in-Robot-Scanned-Workspaces-A-Benchmark-Pipeline/example_datasets/example_small_aruco1"
vrs_file_location = "/home/wmarx/AR-Headset-Localization-in-Robot-Scanned-Workspaces-A-Benchmark-Pipeline/example_datasets/small_aruco1_sitting_20fps.vrs"


robot_data_from_disk = GatheredRobotData.from_folder(robot_data_folder_location)
robot_env_from_robot_data = RobotEnvironment.from_gathered_robot_data(
        robot_data = robot_data_from_disk,
        number_of_sampled_datapoints=10,
        only_sample_robot_datapoints_w_marker_estimates = True,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(),
        est3d_xyz_icp_config=ICPAlignmentConfig()
)

headset_data_from_vrs = HeadsetData.from_vrs_file(vrs_file_location)

### Adding labels to HeadsetData
To add labels to the HeadsetData for accuracy evaluation it has to be joined with an GatheredRobotData object with some supported marker (e.g. aruco/charuco). 
It can then be saved and does not need rebounding for labels (rebounding is still possible), so GatheredRobotData becomes obsolete

In [ ]:
# Labeling by using a robot_env_from_robot_data
labeled_headset_data = create_robot_bound_headset_data(
        headset_data = headset_data_from_vrs,
        robot_data = robot_data_from_disk
    )

### Saving and loading RobotEnvironments and HeadsetData
`RobotEnvironment` and `HeadsetData` both have file interfaces which can be used to load/save them from/to the disk

In [ ]:
processed_datasets_location = "/home/wmarx/AR-Headset-Localization-in-Robot-Scanned-Workspaces-A-Benchmark-Pipeline/processed_datasets"

labeled_headset_data.save(processed_datasets_location, new_name = "quickstart_headset_data")
robot_env_from_robot_data.save(processed_datasets_location, new_name = "quickstart_robot_data")

robot_env_from_disk = RobotEnvironment.from_folder(f"{processed_datasets_location}/quickstart_robot_data")
headset_data_from_disk = HeadsetData.from_folder(f"{processed_datasets_location}/quickstart_headset_data")

### Alternative: Creating from TU-München Dataset
Alternative they cam be created from a TU-München Dataset: https://cvg.cit.tum.de/data/datasets/rgbd-dataset

In [ ]:
from load_from_tum import *

tum_rgbd_dataset_location = "/home/wmarx/AR-Headset-Localization-in-Robot-Scanned-Workspaces-A-Benchmark-Pipeline/datasets/rgbd_dataset_freiburg2_desk"

tum_robot_env, tum_headset_data = robot_environment_and_headset_data_from_tum(
        folder=tum_rgbd_dataset_location,
        rgb_camera_name="freiburg2",
        time_tolerance= 0.01,
        n_robot_images= 20,
        xyz_image_generation_config=XYZImageGenerationConfig(),
        xyz_image_alginment_config=ICPAlignmentConfig(do_alginment=False),
)

### Visualising Datasets
Robot environments and Headset datasets can be visualized in 3d

In [ ]:
vis_robot_env, vis_headset, vis_both = False, False, True

if vis_robot_env:
    robot_env_from_disk.visualize_3d_data()
if vis_headset:
    headset_data_from_disk.visualize_3d_data()
if vis_both:
    visualize_robot_camera_environment_combo(robot_env=robot_env_from_disk, headset_data=headset_data_from_disk)

## Testing Predictors

In [ ]:
from predictor_grader import *
from pose_pred_points import *
from pose_pred_points_lines import *
from pose_pred_points_ellipsoids import *

### Creating Predictors
Now an `PosePredictor` can be created. An `PosePredictor` instance is build upon an `RobotEnvironment` instance and can predict positions from headset-images.

In [ ]:
chosen_headset_data = headset_data_from_disk
chosen_robot_env = robot_env_from_disk

# Simple Point only predictor
point_predictor = OnlyPointsPredictor(
        cam2_intrinsic_mtx=chosen_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=chosen_robot_env.robot_bgr_images,
        cam1_xyz_images=chosen_robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(extract_and_match=ExtractAndLightGlue())
)

### Grading the Performance of an Initialised Predictor:

In [ ]:
init_predictor_grade = PredictionOnDataset(
    predictor = point_predictor,
    headset_data = chosen_headset_data,
    number_retry = 1
)

init_predictor_grade.print_summary()

The predictions can also be visualized in 3d

In [ ]:
visualize_prediction = True
if visualize_prediction:
    init_predictor_grade.visualize_predictions(
        robot_env=chosen_robot_env,
        show_label=False
    )

### Grading of multiple uninitialised Predictors in an Environment
Predictors provide `get_creation_function` methods, which can be used to initialize one, to grade the creation behaviour and time. Their signatures are similar to the of `__init__`. `GradablePosePredictor` provided additional configuration options on how many retries per prediction if `update_pose` should be used etc.

In [ ]:
# Creation of the Predictors
points_light_glue = GradablePosePredictor(
    creator= OnlyPointsPredictor.get_creation_function(
        cam2_intrinsic_mtx=chosen_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(extract_and_match=ExtractAndLightGlue())
    ),
    name="points_light_glue"
)

points_lines_light_glue = GradablePosePredictor(
    creator=LinePredictor.get_creation_function(
        cam2_intrinsic_mtx = chosen_headset_data.intrinsic_cam_mtx,
    ),
    name="points_lines_light_glue"
)

ellipsoids_light_glue = GradablePosePredictor(
    creator=EllipsoidPredictor.get_creation_function(
        cam2_intrinsic_mtx = chosen_headset_data.intrinsic_cam_mtx,
    ),
    name="ellipsoids_light_glue"
)

Now those can be used to create a grader object for multiple `PosePredictor` variants.

In [ ]:
# Initialising the grader:
grader = NPredictors1DatasetGrader(
    gradable_pose_predictors=[points_light_glue, points_lines_light_glue,ellipsoids_light_glue],
    headset_data = chosen_headset_data,
    robot_env = chosen_robot_env,
)
grader.print_summary()